In [ ]:
!pip install torch-geometric


In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)

In [ ]:
!pip install torch-geometric



In [ ]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from scipy.stats import pearsonr
from scipy.stats import pearsonr
from torch_geometric.loader import DataLoader
import random
random.seed(12)
torch.manual_seed(12)
from torch_geometric.loader import DataLoader

In [ ]:
ppi = pd.read_csv("/content/ppi.csv")
labels = pd.read_csv("/content/labels.csv")
expressions = pd.read_csv("/content/expression.csv")
mutations = pd.read_csv("/content/mutations.csv")

In [ ]:
ppi.shape

In [ ]:
ppi_genes = pd.concat([ppi['A'], ppi['B']]).unique()
gene_to_idx = {gene: i for i, gene in enumerate(ppi_genes)}

In [ ]:
edge_src = ppi['A'].map(gene_to_idx).values
edge_dst = ppi['B'].map(gene_to_idx).values

edge_index = torch.tensor(
    np.array([np.concatenate([edge_src, edge_dst]),
              np.concatenate([edge_dst, edge_src])]),
    dtype=torch.long
)


edge_features = torch.tensor(ppi[['combined_score', 'coexpression']].values, dtype=torch.float)
edge_attr = torch.cat([edge_features, edge_features], dim=0)


In [ ]:
expr_no_id = expressions.drop('ModelID', axis=1)
mut_no_id = mutations.drop('ModelID', axis=1)
mutations_aligned = mut_no_id.reindex(columns=ppi_genes, fill_value=0)
labels_no_id = labels.drop('ModelID', axis=1)
labels_reindexed = labels_no_id.reindex(columns=ppi_genes)

In [ ]:
graph_list = []
for i in range(32):
    expr_row = expr_no_id[ppi_genes].iloc[i].values
    mut_row = mutations_aligned.iloc[i].values
    x = torch.tensor(np.column_stack([expr_row, mut_row]), dtype=torch.float)

    y_raw = labels_reindexed.iloc[i]
    cell_mask = torch.tensor((~y_raw.isna().values) & np.isin(ppi_genes, labels_no_id.columns), dtype=torch.bool)
    y = torch.tensor(y_raw.fillna(0).values, dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, mask=cell_mask)
    graph_list.append(data)


In [ ]:
import torch.nn as nn
from torch_geometric.nn import GATConv

In [ ]:
class KidneyGAT(nn.Module):
    def __init__(self):
        super().__init__()
        self.expr_encoder = nn.Linear(1, 32)
        self.mut_encoder = nn.Linear(1, 32)
        self.conv1 = GATConv(64, 64, heads=4, edge_dim=2)
        self.conv2 = GATConv(64 * 4, 64, heads=4, edge_dim=2, concat=False)
        self.relu = nn.LeakyReLU()
        self.dropout = nn.Dropout(0.3)
        self.out = nn.Linear(64, 1)

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        expr = self.expr_encoder(x[:, 0:1])
        mut = self.mut_encoder(x[:, 1:2])
        x = torch.cat([expr, mut], dim=1)

        x = self.conv1(x, edge_index, edge_attr)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index, edge_attr)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.out(x)
        return x.squeeze()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = KidneyGAT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
loss_fn = nn.MSELoss()

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=12)
all_attn = []
fold_pearsons = []

for fold, (train_idx, test_idx) in enumerate(kf.split(graph_list)):
    train_data = [graph_list[i] for i in train_idx]
    test_data = [graph_list[i] for i in test_idx]
    train_loader = DataLoader(train_data, batch_size=1, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=1)

    # Fresh model each fold
    model = KidneyGAT().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    loss_fn = nn.MSELoss()

    # Train
    for epoch in range(200):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            pred = model(batch)
            loss = loss_fn(pred[batch.mask], batch.y[batch.mask])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()



    # Evaluate + extract attention
    model.eval()
    fold_corrs = []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)

            # Pearson
            pred = model(batch)
            p = pred[batch.mask].cpu().numpy()
            a = batch.y[batch.mask].cpu().numpy()
            r, _ = pearsonr(p, a)
            fold_corrs.append(r)

            # Attention extraction
            expr = model.expr_encoder(batch.x[:, 0:1])
            mut = model.mut_encoder(batch.x[:, 1:2])
            x = torch.cat([expr, mut], dim=1)
            x, _ = model.conv1(x, batch.edge_index, batch.edge_attr, return_attention_weights=True)
            x = model.relu(x)
            x = model.dropout(x)
            x, (edge_idx, scores) = model.conv2(x, batch.edge_index, batch.edge_attr, return_attention_weights=True)

            avg_scores = scores.mean(dim=1).cpu().numpy()
            all_attn.append(avg_scores)

    fold_r = np.mean(fold_corrs)
    fold_pearsons.append(fold_r)
    print(f"Fold {fold+1}: Pearson r = {fold_r:.4f}")

print(f"\nMean Pearson r across 5 folds: {np.mean(fold_pearsons):.4f} ± {np.std(fold_pearsons):.4f}")


In [ ]:
# Get edge indices from one sample
sample = graph_list[0].to(device)
with torch.no_grad():
    expr = model.expr_encoder(sample.x[:, 0:1])
    mut = model.mut_encoder(sample.x[:, 1:2])
    x = torch.cat([expr, mut], dim=1)
    x, _ = model.conv1(x, sample.edge_index, sample.edge_attr, return_attention_weights=True)
    x = model.relu(x)
    x = model.dropout(x)
    x, (edge_idx, _) = model.conv2(x, sample.edge_index, sample.edge_attr, return_attention_weights=True)

src = edge_idx[0].cpu().numpy()
dst = edge_idx[1].cpu().numpy()

# Average attention across all folds/cell lines
mean_attn = np.mean(all_attn, axis=0)

# Filter self-loops
mask = src != dst
src_filtered = src[mask]
dst_filtered = dst[mask]
attn_filtered = mean_attn[mask]

idx_to_gene = {i: gene for gene, i in gene_to_idx.items()}

# Top 100 interactions
top_indices = np.argsort(attn_filtered)[-100:][::-1]
print("Top 20 gene interactions (averaged across all folds):\n")
for rank, i in enumerate(top_indices, 1):
    g1 = idx_to_gene[src_filtered[i]]
    g2 = idx_to_gene[dst_filtered[i]]
    print(f"{rank:2d}. {g1:10s} <-> {g2:10s}  attn: {attn_filtered[i]:.4f}")


In [ ]:
known_cancer_genes = ['VHL', 'TP53', 'EGFR', 'PTEN', 'MTOR', 'MYC', 'KRAS', 'BRCA1', 'PIK3CA', 'BRAF']

for gene in known_cancer_genes:
    if gene not in gene_to_idx:
        print(f"{gene} — not in PPI network")
        continue

    idx = gene_to_idx[gene]
    gene_mask = (src_filtered == idx) | (dst_filtered == idx)
    gene_attn = attn_filtered[gene_mask]
    gene_src = src_filtered[gene_mask]
    gene_dst = dst_filtered[gene_mask]

    if len(gene_attn) == 0:
        print(f"{gene} — no edges found")
        continue

    top5 = np.argsort(gene_attn)[-5:][::-1]

    print(f"\n{gene} — {len(gene_attn)} edges, max attn: {gene_attn.max():.4f}")
    for i in top5:
        partner = idx_to_gene[gene_dst[i]] if gene_src[i] == idx else idx_to_gene[gene_src[i]]
        print(f"  <-> {partner:12s}  attn: {gene_attn[i]:.4f}")
